In [8]:
import altair as alt
import pandas as pd
from ecostyles import EcoStyles

styles = EcoStyles()
styles.register_and_enable_theme()

# ---------- settings ----------
GREY = "#676A86"   # ecostyles axis/caption grey
INK = "#2B2D42"    # GDP line

# Services take the theme's first palette colour (teal); fallback if the theme is structured differently
try:
    TEAL = alt.themes.get()()["config"]["range"]["category"][0]
except (KeyError, IndexError, TypeError):
    TEAL = "#00A3A1"

# ---------- data ----------
# ONS contributions to three-month GDP growth (percentage points)
raw = pd.DataFrame(
    {
        "month": ["Sep 25", "Oct 25", "Nov 25", "Dec 25", "Jan 26", "Feb 26", "Mar 26",
                  "Apr 26", "May 26", "Jun 26", "Jul 26", "Aug 26", "Sep 26"],
        "GDP":          [0.2, 0.0, -0.1, 0.0, 0.2, 0.4, 0.6, 0.8, 0.7, 0.5, 0.5, 0.6, 0.7],
        "Services":     [0.29, 0.12, 0.04, -0.02, 0.12, 0.37, 0.61, 0.72, 0.61, 0.50, 0.61, 0.75, 0.77],
        "Production":   [-0.12, -0.09, -0.03, 0.15, 0.18, 0.10, 0.04, 0.05, 0.04, -0.02, -0.10, -0.06, 0.01],
        "Construction": [0.00, -0.02, -0.06, -0.12, -0.12, -0.08, 0.02, 0.08, 0.10, 0.05, -0.01, -0.05, -0.04],
    }
)
month_order = raw["month"].tolist()
df = raw.melt("month", var_name="component", value_name="value")

# ---------- chart ----------
# Colour carries the story: services (teal) vs everything else (greys)
domain = ["Services", "Production", "Construction", "GDP"]
colour = alt.Color(
    "component:N",
    sort=domain,
    scale=alt.Scale(domain=domain, range=[TEAL, "#B8BCC8", "#8A8EA3", INK]),
    legend=alt.Legend(title=None, orient="bottom", direction="horizontal"),
)
x = alt.X("month:O", sort=month_order, title=None, axis=alt.Axis(labelAngle=0))

bars = (
    alt.Chart(df[df["component"] != "GDP"])
    .mark_bar(size=24)
    .encode(x=x, y=alt.Y("value:Q", title="Percentage points"), color=colour)
)

line = (
    alt.Chart(df[df["component"] == "GDP"])
    .mark_line(strokeWidth=1.8, point=alt.OverlayMarkDef(size=40))
    .encode(x=x, y="value:Q", color=colour)
)

zero = alt.Chart(pd.DataFrame({"y": [0]})).mark_rule(color=GREY, strokeWidth=1).encode(y="y:Q")

chart = (
    (zero + bars + line)
    .properties(
        width=700,
        height=270,
        
    )
)

final = (
    alt.vconcat(chart, note, spacing=0)
    .configure(background="#f7f7f7", font="Circular Std")
    .configure_view(fill="transparent", stroke="transparent")
)

final.display()
styles.save(final, path=".", name="uk_gdp_contributions", svg=True)
final.save("uk_gdp_contributions.html")

alt.VConcatChart(...)